## Set Up Notebook

In [ ]:
# Import packages
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

from scipy.stats import chi2_contingency
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Disable scientific notation
pd.set_option('display.float_format', lambda x: '%.3f' % x)

## Clean Data

In [ ]:
# Load data
df = pd.read_csv('speech_lying_deception_data.csv')

# Print results for Control Question 1
print("\nControl Question 1")
print("\nCounts:\n", df['kontrolle1'].value_counts().sort_index())
print("\nPercentages:\n", df['kontrolle1'].value_counts(normalize = True).sort_index() * 100)

# Print results for Control Question 2
print("\nControl Question 2")
print("\nCounts:\n", df['kontrolle2b'].value_counts().sort_index())
print("\nPercentages:\n", df['kontrolle2b'].value_counts(normalize = True).sort_index() * 100)

# Create clean DataFrame
df_clean = df[
    (df['consent']     == 1.0) & 
    (df['lastpage']    == 20) & 
    (df['kontrolle1']  == 2) & 
    (df['kontrolle2b'] == 1)
].copy()

# Define conditions
conditions = {
    1: ('Lying',      'Intent'),
    2: ('Lying',      'No Intent'),
    3: ('Misleading', 'Intent'),
    4: ('Misleading', 'No Intent')
}

# Apply mapping and split into new columns
df_clean['type'], df_clean['intention'] = zip(*df_clean['randnumber'].map(conditions))

# Combine choices
df_clean['choice'] = df_clean[['auswahl1', 'auswahl2', 'auswahl3', 'auswahl4']].bfill(axis = 1).iloc[:, 0]

# Combine scales
df_clean['scale'] = df_clean[['skala1', 'skala2', 'skala3', 'skala4']].bfill(axis = 1).iloc[:, 0]

# Count participants
print("\nParticipants per group")
print(df_clean['randnumber'].value_counts().sort_index())
print(f"\nTotal sample size: {len(df_clean)}")

## Calculate Sociodemographics

In [ ]:
# Define labels
labels_sociodemo = {
    'geschlecht': {
        1: 'Female',
        2: 'Non-Binary',
        3: 'Male'
    },
    'bildung': {
        1: 'Still in School',
        2: 'Lower Secondary Education',
        3: 'Intermediate Secondary Education',
        4: 'Higher Education Entrance Qualification',
        5: 'Completed Vocational Training',
        6: 'Master Craftsman',
        7: 'Bachelor',
        8: 'Master',
       11: 'Doctorate'
    }
}

# Apply labels
df_clean['geschlecht'] = df_clean['geschlecht'].map(labels_sociodemo['geschlecht'])
df_clean['bildung'] = df_clean['bildung'].map(labels_sociodemo['bildung'])

# Display results
print("\nAge")
print(df_clean['alter'].describe().round(3))

print("\nGender")
print((df_clean['geschlecht'].value_counts(normalize = True) * 100).round(3))

print("\nEducation")
print(df_clean['bildung'].value_counts())

## Calculate Descriptive Statistics

In [ ]:
# Define labels
labels_choice = {
    1: '(1) Lie',
    2: '(2) Mislead',
    3: '(3) Both',
    4: '(4) Neither',
    5: '(5) Not Sure'
}

# Apply labels in new column
df_clean['choice_label'] = df_clean['choice'].map(labels_choice)

# Display results for choice
print("\nChoice")
print(df_clean.groupby(['type', 'intention'])['choice_label'].value_counts().sort_index())

# Display results for scale
print("\nScale")
print(df_clean.groupby(['type', 'intention'])['scale'].describe().round(3))

## Run Chi-Squared Test

In [ ]:
# Create contingency table
contingency_table = pd.crosstab([df_clean['type'], df_clean['intention']], df_clean['choice'])

# Display contingency table
print("\nFrequencies")
print(contingency_table)

# Run Chi-Squared Test
chi2, p, dof, expected = chi2_contingency(contingency_table)

# Display results
print("\nChi-Squared Test")
print(f"Chi-Square:         {chi2:.3f}")
print(f"p:                  {p:.3f}")
print(f"Degrees of Freedom: {dof}")

## Run Multinomial Logistic Regression

In [ ]:
# Fit model
model = smf.mnlogit('choice ~ C(type) * C(intention)', data = df_clean).fit()

# Display results
print(model.summary())

## Run ANOVA

In [ ]:
# Fit model
anova_model = ols('scale ~ C(type) * C(intention)', data = df_clean).fit()

# Generate table
anova_table = sm.stats.anova_lm(anova_model, typ = 2)

# Display results
print(anova_table)

## Run ANCOVA

In [ ]:
# Fit model
ancova_model = ols('scale ~ C(type) * C(intention) + alter + C(geschlecht) + C(bildung)', data = df_clean).fit()

# Generate table
ancova_table = sm.stats.anova_lm(ancova_model, typ = 2)

# Display results
print(ancova_table)

## Run Tukey’s Honestly Significant Difference Test

In [ ]:
# Create combined groups
df_clean['combined_group'] = df_clean['type'].astype(str) + " + " + df_clean['intention'].astype(str)

# Run test
tukey = pairwise_tukeyhsd(endog  = df_clean['scale'],
                          groups = df_clean['combined_group'],
                          alpha  = 0.05)

# Display results
print(tukey.summary())

# Plot results
tukey.plot_simultaneous();

## Create Bar Chart

In [ ]:
# Set plot theme
sns.set_theme(style = "whitegrid", context = "talk")

# Set figure size
plt.figure(figsize = (8, 6))

# Create plot
sns.barplot(
    data    = df_clean,
    x       = 'type',
    y       = 'scale',
    hue     = 'intention',
    capsize = 0.1,
    err_kws = {'linewidth': 1.5},
    palette = "Set2"
)

# Set axis labels
plt.ylabel('Scale')
plt.xlabel('Type')

# Set legend
plt.legend(bbox_to_anchor = (1.05, 1), loc = 'upper left')

# Adjust layout
plt.tight_layout()

# Display plot
plt.show()